In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn


  Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.3-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.10.8-cp311-cp311-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp311-cp311-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp311-cp311-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.1.1-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing import StandardScaler


In [3]:
url = r"getflix_synthetic_dataset_v2.csv"
df = pd.read_csv(url)
df.head()

,CustomerID,age,salary_k,tenure_years,number_of_logins,complaints,engagement_score,subscription_type,region,device_type,signup_date,last_active_date,churn
0,CUST_E3C733E6,45.379254,17.420401,0.17,18,3,6.44,Basic,India,Mobile,2020-07-29,2024-05-19,0
1,CUST_9CA1BA76,38.901590,22.964501,10.00,20,1,29.06,Basic,USA,Smart TV,2018-04-12,2024-08-08,0
2,CUST_DAC2CE28,42.141956,17.902013,4.83,30,3,27.63,Basic,India,Web,2021-01-17,2024-07-13,0
3,CUST_270A241D,52.597429,25.766784,6.09,18,0,19.33,Standard,USA,Mobile,2017-08-10,2024-11-03,0
4,CUST_B5384668,34.602866,40.754629,3.56,25,0,20.67,Premium,India,Mobile,2019-08-03,2024-08-23,0


Cleaning data

In [4]:
df.isna().sum()

CustomerID             0
age                    0
salary_k             297
tenure_years         295
number_of_logins       0
complaints             0
engagement_score       0
subscription_type      0
region                 0
device_type            0
signup_date            0
last_active_date       0
churn                  0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.dropna(inplace=True)

In [7]:
df.isna().sum()

CustomerID           0
age                  0
salary_k             0
tenure_years         0
number_of_logins     0
complaints           0
engagement_score     0
subscription_type    0
region               0
device_type          0
signup_date          0
last_active_date     0
churn                0
dtype: int64

In [8]:
df.info()


<class 'pandas.DataFrame'>
Index: 9413 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         9413 non-null   str    
 1   age                9413 non-null   float64
 2   salary_k           9413 non-null   float64
 3   tenure_years       9413 non-null   float64
 4   number_of_logins   9413 non-null   int64  
 5   complaints         9413 non-null   int64  
 6   engagement_score   9413 non-null   float64
 7   subscription_type  9413 non-null   str    
 8   region             9413 non-null   str    
 9   device_type        9413 non-null   str    
 10  signup_date        9413 non-null   str    
 11  last_active_date   9413 non-null   str    
 12  churn              9413 non-null   int64  
dtypes: float64(4), int64(3), str(6)
memory usage: 1.0 MB


In [9]:
df.age = round(df.age)

In [10]:
df.age = df.age.astype(int)

In [11]:
df.salary_k = round(df.salary_k,2)

In [12]:
for col in ['signup_date', 'last_active_date']:
    df[col] = pd.to_datetime(df[col])

In [13]:
df.info()

<class 'pandas.DataFrame'>
Index: 9413 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   CustomerID         9413 non-null   str           
 1   age                9413 non-null   int64         
 2   salary_k           9413 non-null   float64       
 3   tenure_years       9413 non-null   float64       
 4   number_of_logins   9413 non-null   int64         
 5   complaints         9413 non-null   int64         
 6   engagement_score   9413 non-null   float64       
 7   subscription_type  9413 non-null   str           
 8   region             9413 non-null   str           
 9   device_type        9413 non-null   str           
 10  signup_date        9413 non-null   datetime64[us]
 11  last_active_date   9413 non-null   datetime64[us]
 12  churn              9413 non-null   int64         
dtypes: datetime64[us](2), float64(3), int64(4), str(4)
memory usage: 1.0 MB


In [14]:
df['churn'].value_counts()

# there is class imbalance in dataset

churn
0    7422
1    1991
Name: count, dtype: int64

In [15]:
df.describe()

,age,salary_k,tenure_years,number_of_logins,complaints,engagement_score,signup_date,last_active_date,churn
count,9413.000000,9413.000000,9413.000000,9413.000000,9413.000000,9413.000000,9413,9413,9413.000000
mean,38.116222,29.012839,2.840447,20.513014,2.372570,15.722249,2019-12-11 03:51:27.517263,2024-06-28 16:55:10.485498,0.211516
min,17.000000,4.570000,0.000000,4.000000,0.000000,0.970000,2015-01-06 00:00:00,2024-01-03 00:00:00,0.000000
25%,32.000000,18.860000,0.840000,17.000000,1.000000,11.430000,2017-06-19 00:00:00,2024-04-03 00:00:00,0.000000
50%,38.000000,25.510000,2.060000,20.000000,2.000000,14.720000,2019-12-02 00:00:00,2024-06-29 00:00:00,0.000000
75%,44.000000,35.020000,4.040000,23.000000,3.000000,19.060000,2022-06-02 00:00:00,2024-09-22 00:00:00,0.000000
max,75.000000,234.260000,10.000000,102.000000,11.000000,38.660000,2024-12-02 00:00:00,2025-01-01 00:00:00,1.000000
std,8.888630,16.448305,2.571325,6.441651,1.658229,6.043654,NaN,NaN,0.408405


In [16]:
df.select_dtypes(include=np.number).corr()['churn']

age                -0.068967
salary_k           -0.269082
tenure_years       -0.118510
number_of_logins    0.094931
complaints          0.547250
engagement_score   -0.044255
churn               1.000000
Name: churn, dtype: float64

In [17]:
df

,CustomerID,age,salary_k,tenure_years,number_of_logins,complaints,engagement_score,subscription_type,region,device_type,signup_date,last_active_date,churn
0,CUST_E3C733E6,45,17.42,0.17,18,3,6.44,Basic,India,Mobile,2020-07-29,2024-05-19,0
1,CUST_9CA1BA76,39,22.96,10.00,20,1,29.06,Basic,USA,Smart TV,2018-04-12,2024-08-08,0
2,CUST_DAC2CE28,42,17.90,4.83,30,3,27.63,Basic,India,Web,2021-01-17,2024-07-13,0
3,CUST_270A241D,53,25.77,6.09,18,0,19.33,Standard,USA,Mobile,2017-08-10,2024-11-03,0
4,CUST_B5384668,35,40.75,3.56,25,0,20.67,Premium,India,Mobile,2019-08-03,2024-08-23,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,CUST_827AF943,49,30.88,6.72,20,2,25.04,Standard,India,Web,2019-05-26,2024-10-04,0
9996,CUST_92554530,18,45.40,3.15,20,1,15.97,Standard,Germany,Smart TV,2024-03-07,2024-02-25,0
9997,CUST_968DF7AA,34,30.86,4.03,25,1,21.29,Basic,USA,Mobile,2015-05-08,2024-03-02,0
9998,CUST_A5CDF8E8,40,54.82,2.92,12,0,9.22,Premium,Canada,Web,2015-04-03,2024-05-06,0


In [18]:
df.subscription_type.value_counts()

subscription_type
Basic       4583
Standard    3920
Premium      910
Name: count, dtype: int64

In [19]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
df['subscription_type'] = encoder.fit_transform(df[['subscription_type']])

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 9413 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   CustomerID         9413 non-null   str           
 1   age                9413 non-null   int64         
 2   salary_k           9413 non-null   float64       
 3   tenure_years       9413 non-null   float64       
 4   number_of_logins   9413 non-null   int64         
 5   complaints         9413 non-null   int64         
 6   engagement_score   9413 non-null   float64       
 7   subscription_type  9413 non-null   float64       
 8   region             9413 non-null   str           
 9   device_type        9413 non-null   str           
 10  signup_date        9413 non-null   datetime64[us]
 11  last_active_date   9413 non-null   datetime64[us]
 12  churn              9413 non-null   int64         
dtypes: datetime64[us](2), float64(4), int64(4), str(3)
memory usage: 1.0 MB


In [21]:
df.complaints.value_counts()

complaints
2     2314
1     2104
3     1836
4     1108
0     1060
5      544
6      262
7      130
8       40
9       12
10       2
11       1
Name: count, dtype: int64

In [22]:
df

,CustomerID,age,salary_k,tenure_years,number_of_logins,complaints,engagement_score,subscription_type,region,device_type,signup_date,last_active_date,churn
0,CUST_E3C733E6,45,17.42,0.17,18,3,6.44,0.0,India,Mobile,2020-07-29,2024-05-19,0
1,CUST_9CA1BA76,39,22.96,10.00,20,1,29.06,0.0,USA,Smart TV,2018-04-12,2024-08-08,0
2,CUST_DAC2CE28,42,17.90,4.83,30,3,27.63,0.0,India,Web,2021-01-17,2024-07-13,0
3,CUST_270A241D,53,25.77,6.09,18,0,19.33,2.0,USA,Mobile,2017-08-10,2024-11-03,0
4,CUST_B5384668,35,40.75,3.56,25,0,20.67,1.0,India,Mobile,2019-08-03,2024-08-23,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,CUST_827AF943,49,30.88,6.72,20,2,25.04,2.0,India,Web,2019-05-26,2024-10-04,0
9996,CUST_92554530,18,45.40,3.15,20,1,15.97,2.0,Germany,Smart TV,2024-03-07,2024-02-25,0
9997,CUST_968DF7AA,34,30.86,4.03,25,1,21.29,0.0,USA,Mobile,2015-05-08,2024-03-02,0
9998,CUST_A5CDF8E8,40,54.82,2.92,12,0,9.22,1.0,Canada,Web,2015-04-03,2024-05-06,0


In [23]:
df.signup_date.min(),df.signup_date.max()

(Timestamp('2015-01-06 00:00:00'), Timestamp('2024-12-02 00:00:00'))

In [24]:
df.last_active_date.min(),df.last_active_date.max()

(Timestamp('2024-01-03 00:00:00'), Timestamp('2025-01-01 00:00:00'))

In [25]:
reference_date = pd.to_datetime("2025-01-01")

df["Years_from_signup"] = (reference_date - df["signup_date"]).dt.days / 365
df["days_from_last_active_day"] = (reference_date - df["last_active_date"]).dt.days

In [26]:
df.drop(columns=['signup_date','last_active_date'],inplace = True)

In [27]:
df['churn'] =  df.pop('churn')

In [28]:
df

,CustomerID,age,salary_k,tenure_years,number_of_logins,complaints,engagement_score,subscription_type,region,device_type,Years_from_signup,days_from_last_active_day,churn
0,CUST_E3C733E6,45,17.42,0.17,18,3,6.44,0.0,India,Mobile,4.430137,227,0
1,CUST_9CA1BA76,39,22.96,10.00,20,1,29.06,0.0,USA,Smart TV,6.728767,146,0
2,CUST_DAC2CE28,42,17.90,4.83,30,3,27.63,0.0,India,Web,3.958904,172,0
3,CUST_270A241D,53,25.77,6.09,18,0,19.33,2.0,USA,Mobile,7.400000,59,0
4,CUST_B5384668,35,40.75,3.56,25,0,20.67,1.0,India,Mobile,5.419178,131,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,CUST_827AF943,49,30.88,6.72,20,2,25.04,2.0,India,Web,5.608219,89,0
9996,CUST_92554530,18,45.40,3.15,20,1,15.97,2.0,Germany,Smart TV,0.821918,311,0
9997,CUST_968DF7AA,34,30.86,4.03,25,1,21.29,0.0,USA,Mobile,9.660274,305,0
9998,CUST_A5CDF8E8,40,54.82,2.92,12,0,9.22,1.0,Canada,Web,9.756164,240,0


In [29]:
df.columns


Index(['CustomerID', 'age', 'salary_k', 'tenure_years', 'number_of_logins',
       'complaints', 'engagement_score', 'subscription_type', 'region',
       'device_type', 'Years_from_signup', 'days_from_last_active_day',
       'churn'],
      dtype='str')

In [30]:
numeric_cols = ['age','salary_k','tenure_years','number_of_logins',
                'complaints','engagement_score','Years_from_signup','days_from_last_active_day']

categorical_cols = ['subscription_type','region','device_type']

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
X = df[numeric_cols + categorical_cols]
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state= 42)



In [32]:
X_train

,age,salary_k,tenure_years,number_of_logins,complaints,engagement_score,Years_from_signup,days_from_last_active_day,subscription_type,region,device_type
4626,30,20.29,3.05,28,0,19.85,6.293151,180,2.0,USA,Web
2627,44,23.82,0.91,24,0,13.24,0.857534,41,2.0,Canada,Web
9540,40,17.65,2.90,18,6,16.55,6.671233,226,0.0,India,Mobile
9394,41,17.81,1.14,25,1,16.58,8.153425,242,0.0,Canada,Mobile
7145,43,31.96,1.63,21,3,14.78,1.372603,7,0.0,USA,Mobile
...,...,...,...,...,...,...,...,...,...,...,...
6101,40,32.88,1.22,10,2,6.88,8.410959,14,2.0,India,Web
5521,31,17.87,6.28,22,8,23.83,4.197260,252,0.0,USA,Mobile
5733,38,25.44,1.81,23,4,14.90,3.265753,26,0.0,India,Web
920,48,56.25,2.48,14,1,11.22,5.400000,56,1.0,UK,Smart TV


In [33]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Define the preprocessor using ColumnTransformer
# This allows applying different transformers to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# Fit the preprocessor on the training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

# Transform the test data using the fitted preprocessor
X_test_processed = preprocessor.transform(X_test)

# Optional: Convert the processed arrays back to DataFrames if needed
X_train_processed = pd.DataFrame(X_train_processed, columns=preprocessor.get_feature_names_out())
X_test_processed = pd.DataFrame(X_test_processed, columns=preprocessor.get_feature_names_out())

In [34]:
X_train_processed

,num__age,num__salary_k,num__tenure_years,num__number_of_logins,num__complaints,num__engagement_score,num__Years_from_signup,num__days_from_last_active_day,cat__subscription_type_0.0,cat__subscription_type_1.0,cat__subscription_type_2.0,cat__region_Canada,cat__region_Germany,cat__region_India,cat__region_UK,cat__region_USA,cat__device_type_Mobile,cat__device_type_Smart TV,cat__device_type_Web
0,-0.917526,-0.534012,0.080451,1.145223,-1.436135,0.685707,0.429477,-0.062933,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.656732,-0.319054,-0.751491,0.533560,-1.436135,-0.412486,-1.478670,-1.415444,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.206944,-0.694774,0.022137,-0.383934,2.197227,0.137441,0.562201,0.384661,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.319391,-0.685031,-0.662077,0.686476,-0.830575,0.142426,1.082517,0.540346,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.544285,0.176630,-0.471585,0.074813,0.380546,-0.156629,-1.297858,-1.746275,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7525,0.206944,0.232653,-0.630976,-1.607260,-0.225014,-1.469144,1.172923,-1.678163,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7526,-0.805079,-0.681377,1.336139,0.227729,3.408347,1.346949,-0.306275,0.637649,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
7527,-0.017950,-0.220404,-0.401609,0.380645,0.986106,-0.136692,-0.633276,-1.561399,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7528,1.106520,1.655763,-0.141141,-0.995597,-0.830575,-0.748091,0.115941,-1.269490,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [35]:
y_train

4626    0
2627    0
9540    1
9394    0
7145    0
       ..
6101    0
5521    1
5733    0
920     0
7735    0
Name: churn, Length: 7530, dtype: int64

In [36]:
from imblearn.over_sampling import SMOTE

smote = SMOTE()
X_OverSmote, y_OverSmote = smote.fit_resample(X_train_processed, y_train)

In [37]:
y_OverSmote.value_counts()

churn
0    5960
1    5960
Name: count, dtype: int64

In [38]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_OverSmote,y_OverSmote)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [39]:
model.score(X_OverSmote,y_OverSmote)


0.8442953020134228

In [40]:
model.predict(X_test_processed)

array([0, 0, 0, ..., 0, 0, 0], shape=(1883,))

In [41]:
model.score(X_test_processed,y_test)

0.8438661710037175

In [42]:
encoder = OneHotEncoder()
X_train_categorical_encoded = encoder.fit_transform(X_train[['region','device_type']])

In [43]:
X_train_categorical_df = pd.DataFrame(X_train_categorical_encoded.toarray(),
                                      columns=encoder.get_feature_names_out(),
                                      index=X_train.index)
X_train_encoded = pd.concat([X_train[numeric_cols], X_train_categorical_df], axis=1)

In [44]:
from imblearn.over_sampling import SMOTE

smote = SMOTE()
X_train_OverSmote, y_train_OverSmote = smote.fit_resample(X_train_encoded, y_train)

In [45]:
from xgboost import XGBClassifier
xgb =XGBClassifier(
    n_estimators=100,
    learning_rate=0.5,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    enable_categorical=True)

In [46]:
xgb.fit(X_train_OverSmote,y_train_OverSmote)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [47]:
xgb.score(X_train_OverSmote,y_train_OverSmote)

0.9991610738255033

In [48]:
# Transform X_test categorical columns using the same encoder fitted on X_train
X_test_categorical_encoded = encoder.transform(X_test[['region','device_type']])

# Create a DataFrame from the encoded categorical features for X_test
X_test_categorical_df = pd.DataFrame(X_test_categorical_encoded.toarray(),
                                     columns=encoder.get_feature_names_out(),
                                     index=X_test.index)

# Concatenate numeric (including subscription_type which is already float) and encoded categorical features for X_test
X_test_encoded = pd.concat([X_test[numeric_cols], X_test_categorical_df], axis=1)

# Now score the model with the correctly preprocessed X_test
xgb.score(X_test_encoded, y_test)

0.9543281996813595

In [49]:
from sklearn.metrics import recall_score,precision_score,f1_score,accuracy_score

y_pred = xgb.predict(X_test_encoded)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

In [50]:
recall

0.8812351543942993

In [51]:
precision

0.9115479115479116

In [52]:
import joblib
joblib.dump({'model': xgb, 'encoder': encoder}, 'model.pkl')
print('Model saved!')

Model saved!
